## TIFF plot

In [ ]:
# Settings + Data fetching
system("conda install -y conda-forge::r-rcpp conda-forge::openssl conda-forge::r-sf conda-forge::r-terra conda-forge::r-ncdf4")
system("conda install -y conda-forge::r-r.utils conda-forge::r-tidyverse conda-forge::libgdal-hdf5 conda-forge::r-ggplot2")
system("conda install -y conda-forge::r-lubridate conda-forge::r-rcolorbrewer conda-forge::r-lattice conda-forge::r-png")


In [ ]:
library(ncdf4)
library(R.utils)
library(tidyverse)
library(terra)     
library(dplyr)
library(sf)        
library(jsonlite) 
library(utils)
library(ggplot2)
# ---
library(ncdf4) #     ncdf4: open, write and create NetCDF files (also provides metadata information)
library(lubridate) # lubridate: operate on date and times data
library(RColorBrewer) # RColorBrewer: create colour palettes for thematic maps
library(lattice) # lattice : visualization system for typical graphics

In [ ]:
# SET DIRECTORIES
workdir <- getwd()
dataDir <- paste(workdir,"Data",sep = "/")
outputDir <- paste(workdir,"outputs/collection",sep = "/")
scriptDir <- paste(workdir,"scripts",sep = "/")

In [ ]:
# GENERAL FUNCTIONS

# ==============================================================================
# CREATE REPERTORY
# ==============================================================================
create.directory <- function(name.directory){
    ifelse(!dir.exists(file.path(name.directory)),
        dir.create(file.path(name.directory)),
        "Directory Exists")
}

# ==============================================================================
# DOWNLOAD FILENAME
# ==============================================================================
download.file <- function(filename, url){
    options(timeout = 600)  # 10 minutes
    if(file.exists(filename)){
        cat(filename, "is (are) already in your repertory.")
        } else {
        download.file(url, filename, mode = "wb")
        print('File Downloaded')
    }
}

# ==============================================================================
# UNZIP FILENAME
# ==============================================================================
unzip.file <- function(filename){
    # nchar(filename)
    filename.length <- nchar(filename)
    
    # Get the extension ("." + 2 letters)
    # substr(filename,filename.length-2, filename.length)
    
    # Get the name without the extension ("." + 2 letters)
    start <- 1
    end <- filename.length-3
    unzip.filename <- substr(filename,start, end)

    
    if(!file.exists(unzip.filename)){
 
        # print(global_topo_tiff_gz)
        # print(global_topo_tiff)
        
        R.utils::gunzip(filename, overwrite=FALSE, remove=TRUE, BFR.SIZE=1e+07)
        cat(filename, "successfully unzipped!")
    }

    return(unzip.filename)
}

# ==============================================================================
# MOVE TO DATA DIRECTORY 
# ==============================================================================
move.file <- function(filename, new.path){
    new.filename <- paste(new.path, filename, sep = "/")
    file.rename(from=filename, to=new.filename)
    return(new.filename)
}

# ==============================================================================
# FILE INFO
# ==============================================================================
# print(file.info( substr(global_topo_tiff_gz,start, end) ))


# NASA Data - SST

In [ ]:
# 2.2 DOWNLOAD NASA NetCDF ----

# Root directory containing yearly folders (e.g., data/2008, data/2009, ...)
create.directory(dataDir)

# NASA_API.filename <- paste(scriptDir, NASA_API.filename, sep="/")
NASA_API.filename <- "NASA_API.py"

NASA.cmd <- paste("python", NASA_API.filename, sep=" ")

print(NASA.cmd)
ifelse(test = file.exists(NASA_API.filename),
       yes = system(NASA.cmd),
       no = print("Python file does not exist"))

# List all NetCDF files recursively
nc_files <- list.files(path = dataDir,
                       pattern = "\\AQUA_MODIS.*\\.nc$",
                       full.names = TRUE,
                       recursive = TRUE)

In [ ]:
# Function to read ONE file
read_sst_file <- function(file_path) {
  
  nc <- nc_open(file_path)
  
  # Extract variables
  lon <- ncvar_get(nc, "lon")
  lat <- ncvar_get(nc, "lat")
  
  sst_raw <- ncvar_get(nc, "sst", raw_datavals = TRUE)
  
  # Attributes
  fill_value <- ncatt_get(nc, "sst", "_FillValue")$value
  scale      <- ncatt_get(nc, "sst", "scale_factor")$value
  
  # Clean data
  sst_raw[sst_raw == fill_value] <- NA
  sst <- sst_raw * scale
  
  # Convert to dataframe
  coords <- expand.grid(lon = lon, lat = lat)
  df <- cbind(coords, sst = as.vector(sst))
  
  df <- na.omit(df)
  
  # Optional: add time info from filename
  df$file <- basename(file_path)
  
  nc_close(nc)
  
  return(df)
}

In [ ]:
# Apply to all files and concatenate
sst_all <- bind_rows(lapply(nc_files, read_sst_file))
head(sst_all)

In [ ]:
table_filename <- paste(outputDir,"raw_sst_table.tsv", sep="/")
write.table(sst_all, table_filename, row.names=FALSE, sep="\t")


In [ ]:
Antarctic_sst_coords <- sst_all %>%
  group_by(lon, lat) %>%
    summarise(
    mean_sst   = mean(sst, na.rm = TRUE),
        .groups = "drop_last")

In [ ]:
head(Antarctic_sst_coords)
dim(Antarctic_sst_coords)

length(unique(Antarctic_sst_coords$lat)) ; length(unique(sst_all$lat))
length(unique(Antarctic_sst_coords$lon)) ; length(unique(sst_all$lon))


In [ ]:
6811 * 3098
colSums(is.na(sst_all)) ; range(sst_all$sst)
colSums(is.na(Antarctic_sst_coords)) ; range(Antarctic_sst_coords$mean_sst)


In [ ]:
table_filename <- paste(outputDir,"mean_sst_table.tsv", sep="/")
write.table(Antarctic_sst_coords, table_filename, row.names=FALSE, sep="\t")


### Plots

In [ ]:
Bathy <- rast(bathy_file)
print("Converted into Raster File")

In [ ]:
# ==============================================================================
# 3. MAP PLOTTING ----
# ==============================================================================
png <- 1
if (png) {
  png(filename = "outputs/collection/Fig1.png", width = 1080, height = 720)
    
} else {
  pdf("outputs/collection/Fig1.pdf")
}

Depth_cuts <- c(-8200 ,-7000 ,-6000 ,-5000, -4000, -3000, -1800, -1400, -1000,  -600,  -400 , -200  ,   0 ,   50  , 250   ,500)
Depth_cols <- c(
  "#D6EAF8", "#AED6F1", "#85C1E9", "#5DADE2", "#3498DB",
  "#5DADE2", "#85C1E9", "#A9CCE3", "#D4E6F1", "#EBF5FB",
  "#F4F6F7", "#F8F9F9", "#FDFEFE", "#F2F3F4", "#EAEDED"
)

# Plot bathymetry
plot(Bathy,breaks = Depth_cuts, col = Depth_cols, legend = FALSE, axes = FALSE, box = FALSE,mar=c(0,0,0,0))

# Save the plot ---
dev.off()
